# Bengaluru Weather, Air Quality & Bookstore Data Collection

This notebook combines three data sources into a single automated workflow:

1. **Weather Data** — hourly temperature & relative humidity for 5 Bengaluru locations over the past 21 days (Open-Meteo API)
2. **Air Quality Data** — hourly PM10, PM2.5, and CO for the same 5 locations over the past 30 days (Open-Meteo Air Quality API)
3. **Bookstore Data** — title, price, and rating scraped from books.toscrape.com across multiple paginated catalog pages

All outputs are saved as structured CSV files for downstream analytics or modeling.


In [13]:


import requests
import pandas as pd
from bs4 import BeautifulSoup
import time
import os

OUTPUT_DIR = "data"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## Locations

Five representative Bengaluru locations spread across the city.

In [2]:
LOCATIONS = {
    "Bengaluru_Central":  {"lat": 12.9716, "lon": 77.5946},
    "Whitefield":         {"lat": 12.9698, "lon": 77.7499},
    "Electronic_City":    {"lat": 12.8452, "lon": 77.6602},
    "Yelahanka":          {"lat": 13.1005, "lon": 77.5963},
    "Jayanagar":          {"lat": 12.9250, "lon": 77.5938},
}

for name, coords in LOCATIONS.items():
    print(f"{name}: {coords['lat']}, {coords['lon']}")


Bengaluru_Central: 12.9716, 77.5946
Whitefield: 12.9698, 77.7499
Electronic_City: 12.8452, 77.6602
Yelahanka: 13.1005, 77.5963
Jayanagar: 12.925, 77.5938


## 1. Weather Data

Hourly temperature (°C) and relative humidity (%) for each location over the past 21 days,
using the Open-Meteo Forecast API with the `past_days` parameter (works without an API key).

In [14]:
def fetch_weather(name, lat, lon, past_days=21):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m,relative_humidity_2m",
        "past_days": past_days,
        "forecast_days": 1,
        "timezone": "Asia/Kolkata",
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    df = pd.DataFrame({
        "location": name,
        "latitude": lat,
        "longitude": lon,
        "time": data["hourly"]["time"],
        "temperature_2m_C": data["hourly"]["temperature_2m"],
        "relative_humidity_2m_pct": data["hourly"]["relative_humidity_2m"],
    })
    return df


In [15]:
weather_frames = []

for name, coords in LOCATIONS.items():
    print(f"Fetching weather data for {name}...")
    try:
        df = fetch_weather(name, coords["lat"], coords["lon"], past_days=21)
        weather_frames.append(df)
    except requests.exceptions.RequestException as e:
        print(f"  Failed for {name}: {e}")
    time.sleep(0.5)  # be polite to the API

weather_df = pd.concat(weather_frames, ignore_index=True)
print(f"\nTotal weather records: {len(weather_df)}")
weather_df.head()


Fetching weather data for Bengaluru_Central...
Fetching weather data for Whitefield...
Fetching weather data for Electronic_City...
Fetching weather data for Yelahanka...
Fetching weather data for Jayanagar...

Total weather records: 2640


,location,latitude,longitude,time,temperature_2m_C,relative_humidity_2m_pct
0,Bengaluru_Central,12.9716,77.5946,2026-07-30T00:00,22.1,88
1,Bengaluru_Central,12.9716,77.5946,2026-07-30T01:00,21.9,90
2,Bengaluru_Central,12.9716,77.5946,2026-07-30T02:00,21.5,88
3,Bengaluru_Central,12.9716,77.5946,2026-07-30T03:00,21.2,90
4,Bengaluru_Central,12.9716,77.5946,2026-07-30T04:00,21.0,90


In [5]:
weather_path = os.path.join(OUTPUT_DIR, "bengaluru_weather.csv")
weather_df.to_csv(weather_path, index=False)
print(f"Saved weather data to {weather_path}")


Saved weather data to data\bengaluru_weather.csv


## 2. Air Quality Data

Hourly PM10, PM2.5, and carbon monoxide (CO) concentrations for the same five locations
over the past 30 days, using the Open-Meteo Air Quality API.

In [6]:
def fetch_air_quality(name, lat, lon, past_days=30):
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "pm10,pm2_5,carbon_monoxide",
        "past_days": past_days,
        "forecast_days": 1,
        "timezone": "Asia/Kolkata",
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    df = pd.DataFrame({
        "location": name,
        "latitude": lat,
        "longitude": lon,
        "time": data["hourly"]["time"],
        "pm10_ugm3": data["hourly"]["pm10"],
        "pm2_5_ugm3": data["hourly"]["pm2_5"],
        "carbon_monoxide_ugm3": data["hourly"]["carbon_monoxide"],
    })
    return df


In [7]:
air_quality_frames = []

for name, coords in LOCATIONS.items():
    print(f"Fetching air quality data for {name}...")
    try:
        df = fetch_air_quality(name, coords["lat"], coords["lon"], past_days=30)
        air_quality_frames.append(df)
    except requests.exceptions.RequestException as e:
        print(f"  Failed for {name}: {e}")
    time.sleep(0.5)

air_quality_df = pd.concat(air_quality_frames, ignore_index=True)
print(f"\nTotal air quality records: {len(air_quality_df)}")
air_quality_df.head()


Fetching air quality data for Bengaluru_Central...
Fetching air quality data for Whitefield...
Fetching air quality data for Electronic_City...
Fetching air quality data for Yelahanka...
Fetching air quality data for Jayanagar...

Total air quality records: 3720


,location,latitude,longitude,time,pm10_ugm3,pm2_5_ugm3,carbon_monoxide_ugm3
0,Bengaluru_Central,12.9716,77.5946,2026-07-21T00:00,5.6,4.5,233.0
1,Bengaluru_Central,12.9716,77.5946,2026-07-21T01:00,4.7,3.8,186.0
2,Bengaluru_Central,12.9716,77.5946,2026-07-21T02:00,4.6,3.6,158.0
3,Bengaluru_Central,12.9716,77.5946,2026-07-21T03:00,5.0,3.7,148.0
4,Bengaluru_Central,12.9716,77.5946,2026-07-21T04:00,5.6,4.0,155.0


In [8]:
air_quality_path = os.path.join(OUTPUT_DIR, "bengaluru_air_quality.csv")
air_quality_df.to_csv(air_quality_path, index=False)
print(f"Saved air quality data to {air_quality_path}")


Saved air quality data to data\bengaluru_air_quality.csv


## 3. Bookstore Data

Book title, price, and star rating scraped from [books.toscrape.com](https://books.toscrape.com),
a site built specifically for scraping practice. The scraper walks through every paginated
catalog page until no further pages remain.

In [9]:
RATING_WORDS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
BASE_URL = "https://books.toscrape.com/catalogue/page-{}.html"


def parse_books_page(html):
    soup = BeautifulSoup(html, "html.parser")
    books = []
    for article in soup.select("article.product_pod"):
        title = article.h3.a["title"].strip()
        price_text = article.select_one("p.price_color").get_text(strip=True)
        price = float(price_text.replace("£", "").replace("Â", "").strip())
        rating_class = article.select_one("p.star-rating")["class"]  # e.g. ["star-rating", "Three"]
        rating_word = [c for c in rating_class if c != "star-rating"][0]
        rating = RATING_WORDS.get(rating_word)
        books.append({"title": title, "price_gbp": price, "rating": rating})
    return books


In [10]:
all_books = []
page = 1

while True:
    url = BASE_URL.format(page)
    resp = requests.get(url, timeout=30)
    if resp.status_code != 200:
        # No more pages
        break

    books = parse_books_page(resp.text)
    if not books:
        break

    all_books.extend(books)
    print(f"Page {page}: collected {len(books)} books (running total: {len(all_books)})")
    page += 1
    time.sleep(0.3)  # be polite to the server

books_df = pd.DataFrame(all_books)
print(f"\nTotal books scraped: {len(books_df)}")
books_df.head()


Page 1: collected 20 books (running total: 20)
Page 2: collected 20 books (running total: 40)
Page 3: collected 20 books (running total: 60)
Page 4: collected 20 books (running total: 80)
Page 5: collected 20 books (running total: 100)
Page 6: collected 20 books (running total: 120)
Page 7: collected 20 books (running total: 140)
Page 8: collected 20 books (running total: 160)
Page 9: collected 20 books (running total: 180)
Page 10: collected 20 books (running total: 200)
Page 11: collected 20 books (running total: 220)
Page 12: collected 20 books (running total: 240)
Page 13: collected 20 books (running total: 260)
Page 14: collected 20 books (running total: 280)
Page 15: collected 20 books (running total: 300)
Page 16: collected 20 books (running total: 320)
Page 17: collected 20 books (running total: 340)
Page 18: collected 20 books (running total: 360)
Page 19: collected 20 books (running total: 380)
Page 20: collected 20 books (running total: 400)
Page 21: collected 20 books (runn

,title,price_gbp,rating
0,A Light in the Attic,51.77,3
1,Tipping the Velvet,53.74,1
2,Soumission,50.10,1
3,Sharp Objects,47.82,4
4,Sapiens: A Brief History of Humankind,54.23,5


In [11]:
books_path = os.path.join(OUTPUT_DIR, "bookstore_catalog.csv")
books_df.to_csv(books_path, index=False)
print(f"Saved bookstore data to {books_path}")


Saved bookstore data to data\bookstore_catalog.csv


## Summary

All three datasets have been collected and saved as CSV files in the `data/` folder:

| File | Rows | Description |
|---|---|---|
| `bengaluru_weather.csv` | hourly × 21 days × 5 locations | Temperature & relative humidity |
| `bengaluru_air_quality.csv` | hourly × 30 days × 5 locations | PM10, PM2.5, CO |
| `bookstore_catalog.csv` | all catalog books | Title, price, rating |

These files are ready for downstream analytics or modeling (e.g. joining weather and air
quality data by timestamp and location, or analyzing pricing/rating distributions in the
bookstore dataset).

In [12]:
print("Weather rows:", len(weather_df))
print("Air quality rows:", len(air_quality_df))
print("Bookstore rows:", len(books_df))


Weather rows: 2640
Air quality rows: 3720
Bookstore rows: 1000
